# Fast GB Likelihoods: Chunked-Heterodyne and Signal-Het

Computing a Galactic-binary likelihood the naive way -- generate the dense
time-domain waveform, forward-FFT, multiply against inverse PSD -- costs ~`O(N)`
per call where `N ~ 10^6` for a year of LISA data at 10 s sampling. Across an MCMC
with 16 walkers x 10 temperatures x 500 steps that's already ~10^11 floating-point
ops per source per Markov chain. Running the global fit on ~10^4 sources makes the
naive path unaffordable.

`gbgpu` ships **two fast likelihood paths** that compute `<d|h>` and `<h|h>`
without ever building the dense template:

1. **Chunked-heterodyne** (``GBWDMComputations``): builds a sparse FD waveform
   per chunk, heterodynes it onto the source's carrier `f0`, and reads off
   the inner-product contribution per chunk. Matches lisatools direct to
   `mm ~ 1e-9` -- treat it as ground truth.

2. **Signal-het (v2 polyphase)**: a bin-folded approximation that's faster
   than chunked-het when candidates stay near a reference. Lives at the
   function level today (``gbgpu.jax.wdm.signal_het_kernels`` for JAX;
   ``GBComputationGroupWrap.gb_signal_het_*`` for C++). A user-facing class
   wrapper is on the roadmap.

This tutorial walks through both. **Prerequisites**: read
[LISAanalysistools/examples/wdm_transform_tutorial.ipynb](../../LISAanalysistools/examples/wdm_transform_tutorial.ipynb) first --
it covers the WDM domain that both paths operate in.

## Setup

Same imports + grid + injection on both paths.

In [ ]:
import warnings
warnings.simplefilter('ignore', DeprecationWarning)

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from lisatools.detector import ESAOrbits
from lisatools.domains import TDSettings, TDSignal, WDMSettings, WDMSignal
from lisatools.sensitivity import XYZ2SensitivityMatrix
from lisatools.utils.constants import YRSID_SI

from lisatools.response.tdiconfig import TDIConfig
from lisatools.response.tdionfly import GBTDIonTheFly

from gbgpu.gbcomps import GBWDMComputations

### WDM grid + observation window

Sprint canonical: `Nf=1460, Nt=2560` at `dt=10s`. Half-year baseline starts at
`t_start = 0.5 yr` (the LISA orbits use a small spin-up window before that).
Frequency band `[0.1, 35] mHz` covers the GB-rich part of LISA's bandwidth.

In [ ]:
dt = 10.0
Nf, Nt = 1460, 2560
Nobs = Nf * Nt
t_start = int(0.5 * YRSID_SI / dt) * dt
Tobs = Nobs * dt
print(f'Nobs = {Nobs:,} samples  ({Tobs/YRSID_SI:.2f} yr)')

EC = 20  # edge-cut: discard the first/last EC m-pixels (response transient)
wdm_set = WDMSettings(
    Nf, Nt, dt, t0=t_start,
    min_freq=1e-4, max_freq=35e-3,
    min_time=EC * Nf * dt, max_time=(Nt - EC) * Nf * dt,
)
layer_df = wdm_set.layer_df
Nf_active = int(wdm_set.ind_max_f - wdm_set.ind_min_f + 1)
Nt_active = int(wdm_set.Nt_active)
print(f'active band: Nf_active={Nf_active}  Nt_active={Nt_active}  '
      f'layer_df={layer_df*1e3:.4f} mHz')

### Orbits + TDI config

Both paths share the same physical detector setup -- ESA orbits, 2nd-gen TDI.

In [ ]:
orbits = ESAOrbits()
tdi_config = TDIConfig('2nd generation')
print('orbits:', type(orbits).__name__)
print('tdi_config:', type(tdi_config).__name__,
      'nchannels =', tdi_config.nchannels)

### Injection: a SNR~50 GB at f0 = 14.22 mHz

Use `GBTDIonTheFly` to generate the dense TD signal once, then WDM-transform
to get the injection used by both fast paths.

In [ ]:
t_tdi = np.linspace(t_start, t_start + Tobs, 16384)
gb_gen = GBTDIonTheFly(
    t_tdi, Tobs, t_start, 1.0 / dt, 1,
    tdi_config=tdi_config, orbits=orbits, tdi_chan='XYZ',
)

# Verification-binary-style parameters at f0 = 14.22 mHz
f0_inj   = 14.22e-3
fdot_inj = 1.0e-16
phi0_inj = 1.4
inc_inj  = np.pi / 3.0
psi_inj  = 0.7
lam_inj  = 2.1
beta_inj = 0.5
amp_inj  = 6.0e-23
params_inj = np.array(
    [amp_inj, f0_inj, fdot_inj, 0.0, phi0_inj, inc_inj, psi_inj, lam_inj, beta_inj]
)

# Dense TD signal (channel 0 of XYZ, just for the WDM transform)
spline = gb_gen(*[np.array([p]) for p in params_inj],
                convert_to_ra_dec=False, return_spline=True)
t_arr = np.arange(Nobs) * dt + t_start
td_inj = np.asarray(spline.eval_tdi(t_arr))[0]   # (3, Nobs)
print('td_inj shape:', td_inj.shape, 'channel-0 peak |h|:',
      np.abs(td_inj[0]).max())

Transform each channel to WDM. The result `wdm_inj` is a `WDMSignal` with shape
`(3, Nf, Nt)` -- one channel per row.

In [ ]:
td_set = TDSettings(N=Nobs, dt=dt)
wdm_inj = TDSignal(td_inj, settings=td_set).transform(wdm_set)
print('wdm_inj.arr shape:', wdm_inj.arr.shape)

sens_mat = XYZ2SensitivityMatrix(wdm_set, model='scirdv1')
print('sens_mat.invC shape:', np.asarray(sens_mat.invC).shape,
      '(3, 3, Nf_active, Nt_active)')

## Path 1: Chunked-heterodyne via `GBWDMComputations`

`GBWDMComputations` is the canonical user-facing fast-likelihood class.
It's a `FastLISAResponseParallelModule` -- backend dispatch is at construction
time via `force_backend`.

Key constructor knobs:

* `wdm_settings`  -- the WDM grid the data + template live on.
* `t_ref`        -- chunk-axis reference time.
* `Nt_sub`       -- WDM time-pixels per chunk (256 is the validated default).
* `n_pad`        -- chunk overlap (typ. `Nt_sub // 8 = 32`).
* `N_sparse`     -- sparse-FD samples per chunk (256 default).
* `N_cp_sig`, `N_cp_orbit` -- spline cache density. **0 = direct (uncached)**;
  >0 enables the cubic-spline cache for speed at the cost of `~mm 1e-11`
  approximation. Leave at 0 for lisatools-direct equivalence.
* `tdi_type='XYZ'` -- cross-channel inner-product flavor.

See ``gbgpu.gbcomps.GBWDMComputations`` for the full signature.

In [ ]:
chunked = GBWDMComputations(
    wdm_set,
    t_ref=t_start,
    Nt_sub=256,
    n_pad=32,
    N_sparse=256,
    N_cp_sig=0,           # direct path -> matches lisatools direct
    N_cp_orbit=0,
    orbits=orbits,        # MUST match the injection orbits
    tdi_config='2nd generation',
    d_d=0.0,              # source-only return (we add <d|d> externally)
    tdi_type='XYZ',
)
print(f'n_chunks = {chunked.n_chunks}   T_chunk = {chunked.T_chunk:.2e} s')
print(f'resolved_tukey_alpha = {chunked.resolved_tukey_alpha}  '
      f'(auto for N_sparse=256)')

### Build a template with `fill_global_wdm`

`fill_global_wdm(params, templates)` scatters the per-source heterodyne WDM
coefficients onto a global `(nchannels, Nf, Nt)` buffer. Useful when you want
to see the actual template, compare to the data, or feed it into a custom
likelihood.

In [ ]:
# Build the template on a FULL-grid (3, Nf, Nt) buffer; then slice it to
# the same active band wdm_inj.arr lives on. WDMSettings exposes the slice
# specs directly: ind_min_f / ind_max_f (frequency layers) and
# active_slice_t (a Python slice over the time axis).
template_full = np.zeros((3, Nf, Nt), dtype=float)
chunked.fill_global_wdm(
    params_inj.reshape(1, 9), template_full,
    convert_to_ra_dec=False, factors=None,
)

inj_active      = np.asarray(wdm_inj.arr)          # (3, Nf_active, Nt_active)
template_active = template_full[:,
                                 wdm_set.ind_min_f:wdm_set.ind_max_f + 1,
                                 wdm_set.active_slice_t]
print('inj_active     :', inj_active.shape)
print('template_active:', template_active.shape)

# Raw active-band inner products (no PSD weighting -- the validated
# accuracy metric uses the same convention).
ip_dd = float(np.sum(inj_active * inj_active))
ip_dh = float(np.sum(inj_active * template_active))
ip_hh = float(np.sum(template_active * template_active))
mm    = 1.0 - ip_dh / np.sqrt(ip_dd * ip_hh)
print(f'\nraw inner products (active band, no PSD weighting):')
print(f'  <d|d> = {ip_dd:.4e}')
print(f'  <d|h> = {ip_dh:.4e}')
print(f'  <h|h> = {ip_hh:.4e}')
print(f'  mismatch = 1 - <d|h>/sqrt(<d|d><h|h>) = {mm:+.3e}')
print(f'  (expect ~1e-9 -- the lisatools-direct floor at N_cp_sig=0)')

Plot the template's `|w_mn|` next to the injection so you can see the
carrier band lining up:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
extent_x = [0.5, (Nt_active * wdm_set.layer_dt + 0.5 * Tobs) / YRSID_SI]
for ax, arr, title in zip(
    axes,
    [np.abs(inj_active[0]), np.abs(template_active[0])],
    ['|w_mn| injection (channel 0)', '|w_mn| chunked-het template'],
):
    im = ax.pcolormesh(arr, cmap='magma', shading='auto')
    ax.set_title(title)
    ax.set_xlabel('n (time-layer)')
    ax.set_ylabel('m (freq-layer, active band)')
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### Evaluate `<d|h>` and `<h|h>` with `get_ll_wdm`

`get_ll_wdm(params, wdm_holder)` returns the source-only return
(`<d|h> - 0.5*<h|h>`). Add `-0.5 * <d|d>` externally to get the full Gaussian
log-likelihood.

The `wdm_holder` argument is a small duck-type that exposes
`linear_data_arr[0]` and `linear_psd_arr[0]` (1D buffers). The standard
`AnalysisContainer` provides this, but for tutorial clarity we wrap the
full-grid data + invC ourselves.

In [ ]:
class _FullGridWDMHolder:
    '''Minimal duck-type for ``GBWDMComputations.get_ll_wdm``.'''
    def __init__(self, data_full, invC_full):
        self.linear_data_arr = [np.ascontiguousarray(data_full).ravel()]
        self.linear_psd_arr  = [np.ascontiguousarray(invC_full).ravel()]
    def __len__(self):
        return 1

# AC's active-band invC is (3, 3, Nf_active, Nt_active); chunked-het reads it
# flat. Replace NaNs (from out-of-band layers) with 0 so the inner products
# are well-defined.
invC = np.asarray(sens_mat.invC)
invC = np.where(np.isfinite(invC), invC, 0.0)
holder = _FullGridWDMHolder(inj_active, invC)

ll_source_only = chunked.get_ll_wdm(
    params_inj.reshape(1, 9), holder,
    convert_to_ra_dec=False,
    use_layer_groups=True,
    group_band_layers=5,
    margin_layers=0,
)
ll_source_only = float(np.asarray(ll_source_only).ravel()[0])

# Build <d|d> the dense way for reference.
d_d = float(np.einsum('cmn,ckmn,kmn->', inj_active, invC, inj_active,
                       optimize=True))
logL_chunked = ll_source_only - 0.5 * d_d
print(f'<d|d>            = {d_d:.4e}   (= 2 * SNR^2)')
print(f'source-only logL = {ll_source_only:+.4e}')
print(f'full logL (@inj) = {logL_chunked:+.4e}   (expect ~0 at injection)')

### Cluster-strength check: sweep `f0` and verify the likelihood peak

Move `f0` by `~0.5 * layer_df` on either side and confirm `logL` peaks at the
injection point. This is the smoke test for any new fast-likelihood setup.

In [ ]:
f0_offsets = np.linspace(-0.5, 0.5, 11) * layer_df
ll_curve   = np.zeros_like(f0_offsets)
for i, df0 in enumerate(f0_offsets):
    p = params_inj.copy()
    p[1] = f0_inj + df0
    s = chunked.get_ll_wdm(
        p.reshape(1, 9), holder,
        convert_to_ra_dec=False,
    )
    ll_curve[i] = float(np.asarray(s).ravel()[0]) - 0.5 * d_d

plt.figure(figsize=(7, 3.5))
plt.plot(f0_offsets / layer_df, ll_curve, 'o-')
plt.axvline(0.0, ls='--', color='r', alpha=0.5)
plt.xlabel('Df0 / layer_df')
plt.ylabel('logL')
plt.title('chunked-het logL vs f0 offset (peaks at injection)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Path 2: Signal-het (v2 polyphase)

Signal-het is the **faster** alternative. The Python prototype clocks at
~`130x` chunked-het wall-clock at the same accuracy near a reference; the C++ /
JAX implementations live in:

* C++: ``GBComputationGroupWrap.gb_signal_het_get_ll_in_kernel`` and
  ``.gb_signal_het_get_ll_sparse`` (reached through
  ``gbgpu_backend_<flavor>.cgbgpu``).
* JAX: ``gbgpu.jax.wdm.signal_het_kernels.gb_signal_het_get_ll_sparse_jax``
  (mirrors the C++).

Unlike chunked-het, **signal-het is built around a single reference
parameter set** -- it bin-folds the de-rotated data around the reference
and then evaluates candidates as small perturbations. Accuracy degrades as
the candidate moves away from the reference; the `max_r` clip catches
catastrophic departures but a smooth moderate-distance degradation is
expected.

A user-facing class wrapper (`GBSignalHetComputations`) is on the roadmap.
Today the recommended path for signal-het workflows is to copy the bin-fold
+ kernel-call pattern from one of:

* `LISAanalysistools/scripts/gb_chunked_het/compare_signalhet_vs_chunked_mcmc.py`
  (full MCMC harness on top of signal-het, with chunked-het as ground truth).
* `LISAanalysistools/scripts/gb_chunked_het/gb_signal_het_wdm_v2.py`
  (the Python prototype + reference implementation).

### Visualize the signal-het accuracy decay (without re-running it here)

A standalone diagnostic ships in the LAT scripts tree:
[scripts/gb_chunked_het/show_signalhet_accuracy_vs_distance.py](../../LISAanalysistools/scripts/gb_chunked_het/show_signalhet_accuracy_vs_distance.py).
Run it once on your backend to see how `|logL_sh - logL_ch|` grows along each
of the eight sampled-basis parameter axes:

```bash
cd LISAanalysistools/scripts/gb_chunked_het
BACKEND=cpu      python show_signalhet_accuracy_vs_distance.py
BACKEND=cuda12x  python show_signalhet_accuracy_vs_distance.py
```

Output is a 2x4 figure (one panel per parameter) with `|diff|` on a log
left-axis and the absolute `logL_chunked` / `logL_signal` curves on a
linear twin axis. Observed on Mac CPU at SNR=50, f0=14.22 mHz:

| axis    | `|diff|` median | `|diff|` max   |
|---------|------------------|----------------|
| amp     | 1.0e-1           | 1.2e-1         |
| f0      | 3.3e+1           | 2.5e+3         |  <-- worst (bin-fold de-rotates around f0_ref)
| fdot0   | 1.3e-1           | 5.6e-1         |
| phi0    | 1.1e-1           | 2.0e-1         |
| cosinc  | 1.7e-1           | 1.4e+0         |
| psi     | 1.7e-1           | 4.9e-1         |
| lam     | 3.3e+0           | 1.5e+2         |  <-- 2nd worst
| sinbeta | 4.2e-1           | 8.8e+0         |

Most axes stay within a few logL units across +/- 1.5 rad. `f0` and `lam`
are signal-het's weak points -- if your MCMC walkers travel far on those
axes between reference-resets, chunked-het is the safer choice.

## Choosing between the two

| | chunked-het | signal-het |
|---|---|---|
| accuracy vs lisatools direct | `mm ~ 1e-9` (essentially exact) | `mm ~ 1e-9` near reference, degrades with distance |
| speed | ~`1.0x` baseline | ~`130x` faster near reference |
| reference parameter required? | No | Yes (the bin-fold center) |
| user-facing class | `GBWDMComputations` | function-level today; class on roadmap |
| validated via | `gb_chunked_test_script.py`, `gb_chunked_prior_draws.py` | `compare_signalhet_vs_chunked_mcmc.py` |

**Rule of thumb**: use chunked-het for the outer MCMC (broad exploration,
swap_ll, gradient sweeps). Use signal-het inside the hottest sampling loop
where the walker is known to be tightly clustered around a reference --
but periodically re-reference and cross-check with chunked-het.

For both paths, the underlying inner-product math + the WDM domain itself are
documented in [LISAanalysistools/examples/wdm_transform_tutorial.ipynb](../../LISAanalysistools/examples/wdm_transform_tutorial.ipynb).

## See also

* `gbgpu.GBGPU` -- direct GB waveform builder (FD output).
* `gbgpu.gbcomps.GBFDComputations` -- the FD-domain analog of
  `GBWDMComputations`; same fast-likelihood pattern but in the FD instead of
  WDM domain.
* `bbhx/examples/sobbh_tutorial.ipynb` -- SOBBH counterpart that shares the
  same chunked-het infrastructure.